# Build initial molecular configurations for LAMMPS

This example packs **40 CPD + 40 dienophile molecules** into an orthorhombic periodic cell, inspects the geometry, and prepares LAMMPS data plus NPT/NVT/NVE inputs. No model weights or simulation are needed.

Run from the repository root or `examples/`, using the environment with MartiniFuncs, Psience, and McUtils installed. The small existing dump supplies example molecular templates. You can replace them with your own ASE molecules.

The example box is deliberately dilute. Packing produces a starting geometry, not an equilibrated state. The generated run files require your external UMA/fairchem launcher.
UNITS and TIMESTEP_PS are set together in the setup cell. Initial-data export and run templates currently support metal units only; the notebook rejects other styles rather than mislabeling files.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ase import Atoms
from ase.io import read, write
from ase.io.lammpsdata import read_lammps_data

from MartiniFuncs import graph
from MartiniFuncs.lammpsio import read_lammps_frame
from MartiniFuncs.initial_configurations import (
    MoleculeSpec, PackingError, pack_molecules, write_initial_data,
)
from MartiniFuncs.lammps_inputs import render_lammps_input
from MartiniFuncs.visualization import to_display_molecule

ROOT = Path.cwd().resolve()
if not (ROOT / "tests/lammpsio_tests").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "tests/lammpsio_tests").is_dir():
    raise FileNotFoundError("Run from the repository root or examples directory")

UNITS = "metal"  # source units and required units of this export/template workflow
TIMESTEP_PS = 0.0005  # ps per MD step in generated run inputs
if UNITS != "metal":
    raise ValueError("This data-export/run-template workflow currently requires metal units")

OUTPUT_DIR = ROOT / "reports/notebook_initial_configuration"
WRITE_FILES = False  # Set True to export the files shown below.
OVERWRITE = False    # Choose a fresh OUTPUT_DIR or explicitly enable replacement.

## 1. Obtain whole molecular templates

For a self-contained example, extract one molecule of each species from the small fixture. The bond graph identifies molecules; `place_component` reconstructs whole coordinates across periodic boundaries. Each template is a nonperiodic ASE `Atoms` object in **angstrom**.

These geometries are instantaneous MD structures, not optimized reference molecules.

### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


In [ ]:
frame = read_lammps_frame(
    ROOT / "tests/lammpsio_tests/frame_0.lammpstrj", units=UNITS
)
source = frame.atoms
bond_graph = graph.build_bond_graph(source, scale=1.3)
id_to_row = {int(atom_id): row for row, atom_id in enumerate(source.arrays["global_index"])}
templates = {}
for component in graph.connected_components(bond_graph, atom_ids=source.arrays["global_index"]):
    ids, xyz, _ = graph.place_component(source, bond_graph, component, strict=True)
    selected = source[[id_to_row[int(i)] for i in ids]]
    formula = selected.get_chemical_formula()
    if formula in {"C5H6", "C11H9NO2"} and formula not in templates:
        templates[formula] = Atoms(
            selected.get_chemical_symbols(), positions=xyz, masses=selected.get_masses()
        )

molecule_a = templates["C5H6"]
molecule_b = templates["C11H9NO2"]
display(pd.DataFrame([
    {"name": "CPD", "formula": molecule_a.get_chemical_formula(), "atoms": len(molecule_a)},
    {"name": "dienophile", "formula": molecule_b.get_chemical_formula(), "atoms": len(molecule_b)},
]))

To use your own structures, replace the two assignments above, for example:

```python
molecule_a = read("molecule_A.xyz")
molecule_b = read("molecule_B.xyz")
# molecule_c = read("molecule_C.xyz")
```

Input molecules must already be whole and nonperiodic, with coordinates in angstrom. Setting `pbc=False` alone does not repair a molecule split across a periodic boundary.

View the two templates below. ASE-to-Psience conversion explicitly changes angstrom to Bohr; Psience plots them in angstrom. Bonds use Psience's own defaults.

### Mathematics: display units

Psience coordinates are Bohr: $\mathbf r_{\rm Bohr}=\mathbf r_{\mathrm{Angstrom}}/a_0$, $a_0\approx0.529177$ Å. The ASE-to-Psience conversion performs this step.


In [3]:
display(to_display_molecule(molecule_a).plot(backend="x3d", background="white"))
display(to_display_molecule(molecule_b).plot(backend="x3d", background="white"))

## 2. Choose the composition, cell, and packing controls

Add another `MoleculeSpec` for each additional species. The cell lengths and minimum intermolecular **atom-atom** separation are in angstrom. Intramolecular distances are preserved.

The seed makes placement reproducible. Random insertion can fail in dense cells; `PackingError` reports the failure rather than returning an incomplete mixture. Increase the cell size or use a different packing method instead of assuming the requested density is achievable by random insertion.

### Mathematics: packing and generated run parameters
For species $s$, `MOLECULE_FILES` counts (or `N_A`, `N_B`) specify $n_s$ copies with $a_s$ atoms each: $N=\sum_s n_s a_s$. `CELL_ANGSTROM` gives orthorhombic lengths $\mathbf L$ (Å), $V=L_xL_yL_z$. Density is $1.66053906660\sum_i m_i/V$ g/cm³ for masses in amu.

For each template, subtract its COM $\mathbf c$. Propose
$$\mathbf r_i'=(\mathbf r_i-\mathbf c)Q^T+\mathbf t,\qquad t_\alpha\sim U(0,L_\alpha).$$
A normalized four-component standard-Gaussian quaternion supplies a uniform proper rotation $Q$. `PACKING_SEED` fixes random insertion order, rotations and translations; it is not a physical parameter. Accept only if every intermolecular minimum-image distance is at least `MIN_DISTANCE_ANGSTROM` $d_{\min}$:
$$\Delta\mathbf r=\mathbf r_i'-\mathbf r_j-\mathbf L\left\lfloor(\mathbf r_i'-\mathbf r_j)/\mathbf L+\tfrac12\right\rfloor,\qquad\|\Delta\mathbf r\|\ge d_{\min}.$$
The conservative self-image check is $2\max_i\|\mathbf r_i-\mathbf c\|+d_{\min}\le\min L_\alpha$. `MAX_ATTEMPTS_PER_MOLECULE` caps trials per copy; failure raises an error, not a partial composition. The notebook's minimum-distance verification uses nearest-integer rounding; half-box ties change the image sign, not distance.

Store $\mathbf n_i=\lfloor\mathbf r_i'/\mathbf L\rfloor$ and $\mathbf r_i^{\rm wrapped}=\mathbf r_i'-\mathbf n_iH$. `image_flags` recover $\mathbf r_i^{\rm whole}=\mathbf r_i^{\rm wrapped}+\mathbf n_iH$. The round-trip check compares those whole coordinates (Å), masses and elements. `MOLECULE_TO_VIEW` selects a molecule ID, not a computed coordinate.

Generated inputs have duration `STEPS`$\times$`TIMESTEP_PS`; default dump spacing is $1000\times$`TIMESTEP_PS`. `TEMPERATURE_K` is the target temperature (NVE: initialization only), `PRESSURE_BAR` the NPT target; $10^4$ bar = 1 GPa. The supplied damping times 0.05 ps and 0.5 ps correspond to $0.05/h$ and $0.5/h$ integration steps; they are control times, not equilibration durations. `VELOCITY_SEED` controls Gaussian initialization; emitted `mom yes rot yes` removes initial bulk momentum/rotation. The renderer delegates dynamics to LAMMPS and the external UMA driver. Its `v_netforce` dump field is $\sqrt{F_x^2+F_y^2+F_z^2}$ **per atom**, not $\|\sum_i\mathbf F_i\|$.


In [9]:
N_A = 40
N_B = 40
CELL_ANGSTROM = [40, 40, 40]
PACKING_SEED = 24673
MIN_DISTANCE_ANGSTROM = 2.0
MAX_ATTEMPTS_PER_MOLECULE = 5000

species = [
    MoleculeSpec("CPD", molecule_a, N_A),
    MoleculeSpec("dienophile", molecule_b, N_B),
    # MoleculeSpec("solvent", molecule_c, 100),
]
packed = pack_molecules(
    species,
    cell=CELL_ANGSTROM,
    seed=PACKING_SEED,
    min_distance=MIN_DISTANCE_ANGSTROM,
    max_attempts_per_molecule=MAX_ATTEMPTS_PER_MOLECULE,
)
density = packed.get_masses().sum() * 1.66053906660 / packed.get_volume()
print(f"{len(packed)} atoms; {sum(s.count for s in species)} molecules")
print(f"Density: {density:.5f} g/cm^3")
print("Placement trials:", packed.info["placement_attempts"])
display(pd.DataFrame([
    {"species": s.name, "molecules": s.count, "atoms_per_molecule": len(s.atoms)}
    for s in species
]))

1360 atoms; 80 molecules
Density: 0.26289 g/cm^3
Placement trials: 184


,species,molecules,atoms_per_molecule
0,CPD,40,11
1,dienophile,40,23


## 3. Inspect periodic packing and atom identity

Packed positions lie in the primary box. `image_flags` recover the whole-molecule coordinates, while `molecule_id` identifies each copy. `template_atom_index` maps each atom back to its source molecule.

The following independent check includes minimum-image contacts between different molecules. It deliberately excludes intramolecular bonds.

In [10]:
lengths = packed.cell.lengths()
minimum_distance = float("inf")
molecule_ids = np.unique(packed.arrays["molecule_id"])
for molecule_id in molecule_ids:
    own = packed.arrays["molecule_id"] == molecule_id
    if own.all():
        continue
    displacement = packed.positions[own][:, None, :] - packed.positions[~own][None, :, :]
    displacement -= lengths * np.rint(displacement / lengths)
    minimum_distance = min(minimum_distance, float(np.linalg.norm(displacement, axis=-1).min()))
if len(molecule_ids) > 1:
    print(f"Minimum intermolecular separation: {minimum_distance:.4f} angstrom")
    assert minimum_distance >= MIN_DISTANCE_ANGSTROM - 1e-10

whole_positions = packed.positions + packed.arrays["image_flags"] @ packed.cell.array
display(pd.DataFrame({
    "local_row": np.arange(len(packed)),
    "lammps_id": packed.arrays["lammps_id"],
    "molecule_id": packed.arrays["molecule_id"],
    "species": packed.arrays["molecule_name"],
    "template_atom_index": packed.arrays["template_atom_index"],
}).head(12))

Minimum intermolecular separation: 2.0091 angstrom


,local_row,lammps_id,molecule_id,species,template_atom_index
0,0,1,1,dienophile,0
1,1,2,1,dienophile,1
2,2,3,1,dienophile,2
3,3,4,1,dienophile,3
4,4,5,1,dienophile,4
5,5,6,1,dienophile,5
6,6,7,1,dienophile,6
7,7,8,1,dienophile,7
8,8,9,1,dienophile,8
9,9,10,1,dienophile,9


The first view shows all atoms wrapped into the simulation box. Bonds are hidden **only for this full-cell view**. The second view reconstructs one whole molecule and retains Psience's inferred bonds. This distinguishes a periodic wrapping artifact from broken molecular geometry.

In [11]:
display(to_display_molecule(packed).plot(
    backend="x3d", background="white", bonds=False
))

MOLECULE_TO_VIEW = int(molecule_ids[0])
mask = packed.arrays["molecule_id"] == MOLECULE_TO_VIEW
single_molecule = packed[mask]
single_molecule.positions = whole_positions[mask]
single_molecule.pbc = False
display(to_display_molecule(single_molecule).plot(
    backend="x3d", background="white"
))

## 4. Prepare NPT, NVT, and NVE inputs

These are **three independent fresh-start examples**, not a sequential protocol. All initialize velocities. NPT controls temperature and isotropic pressure; NVT controls temperature at fixed volume; NVE has neither thermostat nor barostat, so its temperature parameter only sets the initial velocities.

Units are LAMMPS `metal`: timestep/damping in ps, temperature in K, and pressure in bar. The 400 K / 25000 bar example follows the collaboration's settings; it is not a validated compression schedule for the dilute starting box.

The input files require the existing external fairchem driver with **uma-s-1p1 / omol**. Model path, inference settings, and charge/spin belong in that launcher. No potential or model is loaded here. `run` remains the last executable command.

In [ ]:
# This exact type order is shared by data export and dump_modify element.
TYPE_SYMBOLS = sorted(set(packed.get_chemical_symbols()))
TEMPERATURE_K = 400.0
PRESSURE_BAR = 25000.0  # 2.5 GPa; NPT only
STEPS = 100000          # 50 ps; example duration, not an equilibration criterion
VELOCITY_SEED = 4928459

input_templates = {}
for ensemble in ("npt", "nvt", "nve"):
    input_templates[ensemble] = render_lammps_input(
        data_file="initial.data",
        type_symbols=TYPE_SYMBOLS,
        ensemble=ensemble,
        steps=STEPS,
        force_source="external_uma",
        output_prefix=ensemble,
        timestep_ps=TIMESTEP_PS,
        temperature_K=TEMPERATURE_K,
        pressure_bar=PRESSURE_BAR if ensemble == "npt" else None,
        temperature_damping_ps=0.05,
        pressure_damping_ps=0.5,
        initialize_velocities=True,
        velocity_seed=VELOCITY_SEED,
    )
print(input_templates["npt"])

For a new run starting from equilibrated data that **already contain velocities**, set `initialize_velocities=False` and change `data_file`. This preserves the loaded velocities; it does not restore thermostat/barostat state as an exact restart would.

## 5. Export and verify

Set `WRITE_FILES=True` in the first cell, then rerun this cell. Existing files are protected unless `OVERWRITE=True`.

- `initial.data`: atomic-style LAMMPS data, masses and image flags, no velocities or bonded force field.
- `initial.data.json`: molecular identity and type mapping. Atomic-style LAMMPS data have no molecule-ID column; the trajectory reader does not automatically reload this sidecar.
- `initial.extxyz`: ASE geometry and per-atom metadata.
- `npt.in`, `nvt.in`, `nve.in`: independent external-driver run templates.

Use the output directory as the run directory so relative data paths resolve. This notebook writes inputs only; it never launches a simulation.

In [ ]:
if WRITE_FILES:
    filenames = ["initial.data", "initial.data.json", "initial.extxyz", "npt.in", "nvt.in", "nve.in"]
    existing = [OUTPUT_DIR / name for name in filenames if (OUTPUT_DIR / name).exists()]
    if existing and not OVERWRITE:
        raise FileExistsError(f"Choose a new OUTPUT_DIR or set OVERWRITE=True: {existing}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    metadata = write_initial_data(
        packed, OUTPUT_DIR / "initial.data",
        type_symbols=TYPE_SYMBOLS, overwrite=OVERWRITE,
    )
    write(OUTPUT_DIR / "initial.extxyz", packed)
    for ensemble, text in input_templates.items():
        (OUTPUT_DIR / f"{ensemble}.in").write_text(text, encoding="utf-8")

    # ASE reads image flags to recover whole coordinates.
    from ase.data import atomic_numbers
    restored = read_lammps_data(
        OUTPUT_DIR / "initial.data", atom_style="atomic", units=UNITS,
        Z_of_type={i + 1: atomic_numbers[s] for i, s in enumerate(TYPE_SYMBOLS)},
    )
    np.testing.assert_allclose(restored.positions, whole_positions, atol=1e-10)
    np.testing.assert_allclose(restored.get_masses(), packed.get_masses())
    assert restored.get_chemical_symbols() == packed.get_chemical_symbols()
    print("Data round-trip checks passed.")
    print("Written:", OUTPUT_DIR)
    display(pd.DataFrame({"file": filenames}))
else:
    print("Preview complete. Set WRITE_FILES=True to export to:", OUTPUT_DIR)

## Next steps

Inspect molecular geometry and packing, select a physically justified relaxation/compression protocol, then run it through the UMA launcher. Use the trajectory and statistics notebooks to check the resulting forces, temperature, density, structure, and sampling.

Current packing supports neutral templates and orthorhombic periodic cells. It does not optimize molecular geometry, minimize energy, perform dense packing, or establish equilibration.